In [ ]:
!pip install faiss-cpu sentence-transformers groq

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

In [ ]:
docs = [
    "Machine learning is a subfield of AI",
    "FAISS is a library for efficient similarity search",
    "OpenAI provides large language models like GPT-4 and GPT-5",
    "Retrieval-Augmented Generation combines vector search with LLMs",
    "LangChain simplifies building LLM-powered applications"
]



In [ ]:
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")


In [ ]:
doc_embeddings = embedding_model.encode(docs).astype("float32")

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)



In [ ]:
query = "What is the use of FAISS in AI?"

In [ ]:
query_embedding = embedding_model.encode([query]).astype("float32")

In [ ]:
top_k = 2
distances, indices = index.search(query_embedding, top_k)

In [ ]:
retrieved_chunks = [docs[i] for i in indices[0]]

In [ ]:
print("Retrieved Chunks:")
for chunk in retrieved_chunks:
    print("-", chunk)

In [ ]:
context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""


In [ ]:
client = Groq(api_key="YOUR_API_KEY_HERE")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2
)

print("\nFinal Answer:")
print(response.choices[0].message.content)